# EDA — March Machine Learning Mania 2026

**Project goal:** Predict P(Team A beats Team B) for NCAA tournament matchups; optimize for log loss.

**Data sources:** Kaggle CSVs in `data/raw/` (or project root): teams, seasons, regular-season and tournament results (compact/detailed), seeds, slots, optional Massey ordinals and spellings.

**Planned EDA:**
- Distributions of game scores, margins, and season coverage
- Missing values and join keys (Season, TeamID)
- Number of games per season and per team
- Seed and bracket structure sanity checks

In [3]:
import pandas as pd
from pathlib import Path

# Project root is parent of notebooks/
ROOT = Path('..')
DATA_RAW = ROOT / "data" / "raw" if (ROOT / "data" / "raw").exists() else ROOT

teams = pd.read_csv(DATA_RAW / "MTeams.csv")
print("MTeams shape:", teams.shape)
teams.head()

MTeams shape: (381, 4)


,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2026
1,1102,Air Force,1985,2026
2,1103,Akron,1985,2026
3,1104,Alabama,1985,2026
4,1105,Alabama A&M,2000,2026


Print Team Data

In [8]:
print("HEAD: " ,teams.head())
print("DTYPES: " ,teams.dtypes)
print("DESCRIBE: " ,teams.describe())
print("ISNA: " ,teams.isna().sum())
print("NUNIQUE: " ,teams.nunique())
print("COLUMNS: " ,teams.columns)


HEAD:     TeamID     TeamName  FirstD1Season  LastD1Season
0    1101  Abilene Chr           2014          2026
1    1102    Air Force           1985          2026
2    1103        Akron           1985          2026
3    1104      Alabama           1985          2026
4    1105  Alabama A&M           2000          2026
DTYPES:  TeamID            int64
TeamName         object
FirstD1Season     int64
LastD1Season      int64
dtype: object
DESCRIBE:              TeamID  FirstD1Season  LastD1Season
count   381.000000     381.000000    381.000000
mean   1291.000000    1989.713911   2024.913386
std     110.129469       9.919653      5.857101
min    1101.000000    1985.000000   1985.000000
25%    1196.000000    1985.000000   2026.000000
50%    1291.000000    1985.000000   2026.000000
75%    1386.000000    1987.000000   2026.000000
max    1481.000000    2026.000000   2026.000000
ISNA:  TeamID           0
TeamName         0
FirstD1Season    0
LastD1Season     0
dtype: int64
NUNIQUE:  TeamID       

Retrieving regular season games feature set

In [9]:
games = pd.read_csv(DATA_RAW / "MRegularSeasonDetailedResults.csv")
print("MRegularSeasonDetailedResults shape:", games.shape)
games.head()

MRegularSeasonDetailedResults shape: (124529, 34)


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14


Using canonical long-format game results helper from `src.game_results.make_long_regular_season_results` to inspect per-team game rows (one row per team per game).

In [10]:
import sys
sys.path.insert(0, "..")

from src.game_results import make_long_regular_season_results

long_games = make_long_regular_season_results(games)
print("Long-format regular-season rows:", long_games.shape)
long_games.head()

,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_fga3,opp_ftm,opp_fta,opp_or,opp_dr,opp_ast,opp_to,opp_stl,opp_blk,opp_pf
0,2003,10,1104,1328,1,68,62,27,58,3,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,1393,1,70,63,26,62,8,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,1437,1,73,61,24,58,8,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,1457,1,56,50,18,38,3,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,1208,1,77,71,30,61,6,...,16,17,27,21,15,12,10,7,1,14


,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_fga3,opp_ftm,opp_fta,opp_or,opp_dr,opp_ast,opp_to,opp_stl,opp_blk,opp_pf
0,2003,10,1328,1104,0,62,68,22,53,2,...,14,11,18,14,24,13,23,7,1,22
1,2003,10,1393,1272,0,63,70,24,67,6,...,20,10,19,15,28,16,13,4,4,18
2,2003,11,1437,1266,0,61,73,22,73,3,...,18,17,29,17,26,15,10,5,2,25
3,2003,11,1457,1296,0,50,56,18,49,6,...,9,17,31,6,19,11,12,14,2,18
4,2003,11,1208,1400,0,71,77,24,62,6,...,14,11,13,17,22,12,14,4,4,20
